In [ ]:
import os
import urllib.request
import zipfile
import pandas as pd


# Local Directory Setup

In [ ]:

# Local Directory Setup 
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
os.makedirs(DATA_DIR, exist_ok=True)
print(f"✅ Local Data Directory set to: {DATA_DIR}")

# CRITICAL: GitHub Security (.gitignore)
gitignore_path = os.path.join(os.getcwd(), '.gitignore')
with open(gitignore_path, 'a+') as f:
    f.seek(0)
    content = f.read()
    if 'TFE_Data/' not in content:
        f.write("\n# Ignore Dataset Folder\nTFE_Data/\n")
        f.write("__pycache__/\n*.npy\n*.zip\n")
print("✅ .gitignore security is active.")

# Download & Parse Logic
def download_and_extract(url, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    zip_path = os.path.join(extract_to, "temp.zip")
    
    if not os.listdir(extract_to): 
        print(f"Downloading from {url}...")
        urllib.request.urlretrieve(url, zip_path)
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        os.remove(zip_path)
        print("Extraction complete.")
    else:
        print(f"Files already exist in {extract_to}. Skipping download.")


# Flikr8k

In [ ]:
def build_flickr8k(subset_size=2000):
    print("\n--- BUILDING FLICKR8K ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr8k")
    img_dir = os.path.join(dataset_dir, "Images")
    txt_dir = os.path.join(dataset_dir, "Text")
    
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip", img_dir)
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip", txt_dir)
    
    actual_images_dir = os.path.join(img_dir, "Flicker8k_Dataset")
    token_path = os.path.join(txt_dir, "Flickr8k.token.txt")
    data = []

    print("Verifying physical files and syncing captions...")
    with open(token_path, 'r') as f:
        for line in f.read().splitlines():
            if not line: continue
            parts = line.split('\t')
            if len(parts) == 2:
                img_name = parts[0].split('#')[0]
                full_path = os.path.join(actual_images_dir, img_name)
                
                if os.path.exists(full_path):
                    data.append({
                        "image_name": img_name, 
                        "image_path": full_path, 
                        "caption": parts[1].strip()
                    })

    df_flat = pd.DataFrame(data)
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index()
    df.rename(columns={'caption': 'captions'}, inplace=True)
    
    subset_df = df.head(subset_size).copy()
    save_path = os.path.join(DATA_DIR, 'subset_df_Flickr8k.pkl')
    subset_df.to_pickle(save_path)
    
    print(f"Flickr8k Ready! {len(subset_df)} images secured.")
    return "Flickr8k"

# Push to GitHub

In [ ]:
def push_to_github(dataset_name):
    print("\n--- PUSHING TO GITHUB ---")
    commit_msg = f"feat: Add ETL pipeline and .gitignore for {dataset_name}"
    
    os.system("git add .")
    os.system(f'git commit -m "{commit_msg}"')
    os.system("git push origin main")
    
    print("="*50)
    print(f"SUCCESS: Code pushed to GitHub with message: '{commit_msg}'")
    print("="*50)


# Execution of Downloading Flikr8k

In [ ]:
if __name__ == "__main__":
    ds_name = build_flickr8k(subset_size=2000)
    push_to_github(ds_name)